## Import packages

In [12]:
# Needed this for loading the dataset locally
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from datasets import load_dataset
import numpy as np

from collections import Counter

import nltk

In [2]:
# dataset = load_dataset("coastalcph/tydi_xor_rc")

## Load Dataset

In [3]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

## Filter and split Dataset

In [4]:
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()



#filter data 
langlst = ['ko','ar','te']
# Source - https://stackoverflow.com/a/59275490
# Posted by atinjanki
# Retrieved 2026-09-05, License - CC BY-SA 4.0

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]



In [5]:
# --------------------------- What is this for? --------------------------- 

# df_train_ar.duplicated().value_counts()
# df_train_ko.duplicated().value_counts()
# df_train_te.duplicated().value_counts()

print(f"{df_train_ar.duplicated().value_counts()}")
print(f"{df_train_ko.duplicated().value_counts()}")
print(f"{df_train_te.duplicated().value_counts()}")

# --------------------------- What is this for? --------------------------- 

False    2558
Name: count, dtype: int64
False    2412
True       10
Name: count, dtype: int64
False    1355
Name: count, dtype: int64


In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")



def get_statistics(df, split, language):
    question_lengths = []
    context_lengths = []

    for question in df["question"]:
        question_lengths.append(len(tokenizer.tokenize(question)))
    
    for context in df["context"]:
        context_lengths.append(len(tokenizer.tokenize(context)))

    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])


    return {
            "split": split,
            "language": language,
            "n": len(df),
            "answerable_%": 100 * answerable_n / len(df),
            "unanswerable_%": 100 * unanswerable_n / len(df),
            "question_median": np.median(question_lengths),
            "question_IQR": np.percentile(question_lengths, 75) - np.percentile(question_lengths, 25),
            "context_median": np.median(context_lengths),
            "context_IQR": np.percentile(context_lengths, 75) - np.percentile(context_lengths, 25)
        }


statistics = pd.DataFrame([
    get_statistics(df_train_ar, "train", "ar"),
    get_statistics(df_train_ko, "train", "ko"),
    get_statistics(df_train_te, "train", "te"),
    get_statistics(df_validation_ar, "validation", "ar"),
    get_statistics(df_validation_ko, "validation", "ko"),
    get_statistics(df_validation_te, "validation", "te")
])

statistics.round(1)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


,split,language,n,answerable_%,unanswerable_%,question_median,question_IQR,context_median,context_IQR
0,train,ar,2558,90.0,10.0,13.0,6.0,123.0,94.0
1,train,ko,2422,97.4,2.6,14.0,4.0,115.0,87.0
2,train,te,1355,96.7,3.3,18.0,6.5,114.0,87.0
3,validation,ar,415,87.5,12.5,12.0,6.0,118.0,94.5
4,validation,ko,356,94.7,5.3,14.0,4.0,112.5,97.0
5,validation,te,384,75.8,24.2,19.5,6.0,142.0,88.2


In [7]:
overall_rows = [] # All languages, not just ar, ko, te

for split, df in [("train", df_train), ("validation", df_validation)]:
    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    overall_rows.append({
        "split": split,
        "n": len(df),
        "answerable_%": 100 * answerable_n / len(df),
        "unanswerable_%": 100 * unanswerable_n / len(df)
    })

pd.DataFrame(overall_rows).round(1)

,split,n,answerable_%,unanswerable_%
0,train,15343,91.0,9.0
1,validation,3011,76.8,23.2


In [8]:
data_quality_rows = []

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te)
]

for split, language, df in dataframes:
    missing_values = df.isna().sum().sum()
    duplicate_pairs = df[["question", "context"]].duplicated().sum()

    data_quality_rows.append({
        "split": split,
        "language": language,
        "missing_values": missing_values,
        "duplicate_question_context_pairs": duplicate_pairs
    })

pd.DataFrame(data_quality_rows)


,split,language,missing_values,duplicate_question_context_pairs
0,train,ar,2558,0
1,train,ko,2422,10
2,train,te,1305,0
3,validation,ar,415,0
4,validation,ko,356,0
5,validation,te,284,0


In [14]:
def most_common_tokens(df):
    tokens = []

    for question in df["question"]:
        tokens.extend(tokenizer.tokenize(question))

    return Counter(tokens).most_common(5)


ar_top5 = most_common_tokens(df_train_ar)
ko_top5 = most_common_tokens(df_train_ko)
te_top5 = most_common_tokens(df_train_te)

print("Arabic:", ar_top5)
print("Korean:", ko_top5)
print("Telugu:", te_top5)


Arabic: [('؟', 2556), ('ال', 2269), ('م', 892), ('في', 624), ('من', 616)]
Korean: [('?', 2420), ('##가', 2219), ('##인', 1495), ('##는', 988), ('##은', 949)]
Telugu: [('?', 1355), ('ఎ', 882), ('##వ', 544), ('##ా', 515), ('ప', 505)]


In [15]:
span_rows = []

df_train_selected = df_train[df_train["lang"].isin(["ar", "ko", "te"])]
df_validation_selected = df_validation[df_validation["lang"].isin(["ar", "ko", "te"])]

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("train", "all", df_train_selected),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te),
    ("validation", "all", df_validation_selected)
]

for split, language, df in dataframes:
    answerable = df[df["answerable"] == True]
    failures = 0

    for context, start, answer in zip(
        answerable["context"],
        answerable["answer_start"],
        answerable["answer"]
    ):
        start = int(start)

        if context[start:start + len(answer)] != answer:
            failures += 1

    span_rows.append({
        "split": split,
        "language": language,
        "checked": len(answerable),
        "failures": failures
    })

pd.DataFrame(span_rows)


,split,language,checked,failures
0,train,ar,2303,0
1,train,ko,2359,0
2,train,te,1310,0
3,train,all,5972,0
4,validation,ar,363,0
5,validation,ko,337,0
6,validation,te,291,0
7,validation,all,991,0


In [16]:
df_train_selected = df_train[df_train["lang"].isin(["ar", "ko", "te"])]

answerable_n = len(df_train_selected[df_train_selected["answerable"] == True])
unanswerable_n = len(df_train_selected[df_train_selected["answerable"] == False])

print("Answerable:", answerable_n)
print("Unanswerable:", unanswerable_n)

if answerable_n >= unanswerable_n:
    majority_label = True
else:
    majority_label = False

print("Majority prediction:", majority_label)



Answerable: 5972
Unanswerable: 363
Majority prediction: True
